In [1]:
from notebooks.models.utility import load_and_validate_final_data
from tauso.data.consts import INHIBITION, CANONICAL_GENE, CELL_LINE
import pandas as pd
import numpy as np
import cupy as cp
import xgboost as xgb
from scipy.stats import spearmanr

# 1. Load Data
final_data, all_features = load_and_validate_final_data(version='oligo', load_competition=True)
test_df = final_data[final_data['split'] == 'test'].copy()

X_test_np = test_df[all_features].values
y_test_np = test_df[INHIBITION].values

# Transfer to GPU
print("Transferring full matrices to GPU (cupy)...")
X_test_gpu = cp.array(X_test_np)
print("GPU Transfer Complete.")

Transferring full matrices to GPU (cupy)...
GPU Transfer Complete.


In [2]:
from pathlib import Path
import xgboost as xgb

seen_model_dir = Path('/home/michael/career/tauso_article/tauso_source2/notebooks/models/SeenOligoModel')

# 1. Define models to evaluate: { "Display Name": "filename.json" }
model_files = {
    'Baseline': 'ModelOligoGlobalL2.json',
    'BaselineCustomId': 'ModelOligoGlobalL2CustomId.json',
    'GA_Model': 'ModelOligoGlobalL2_GA.json',
    'L1_Model': 'ModelOligoGlobalL1.json',
    'Residual': 'GlobalResidualModel.json',
    'ResidualGA': 'GlobalResidualModelGA.json',
    # 'Another_Model': 'ModelOligo_V3.json'  <-- Just add new models here
}

models = {}
test_predictions = {}

print("Loading Models and Generating Predictions...\n")

for model_name, filename in model_files.items():
    print(f"⚙️ Processing {model_name}...")

    # 2. Load Model
    booster = xgb.Booster()
    booster.load_model(str(seen_model_dir / filename))
    models[model_name] = booster

    # 3. Extract specific features required by this model
    feature_names = booster.feature_names
    feature_idxs = [all_features.index(f) for f in feature_names]

    # 4. Subset GPU data and predict
    X_test_gpu_subset = X_test_gpu[:, feature_idxs]
    preds = booster.inplace_predict(X_test_gpu_subset)

    if hasattr(preds, 'get'):
        preds = preds.get()

    if 'Residual' in model_name:
        # Assuming you have the base model predictions or 'oligo_ai_score' loaded
        base_preds = test_df['oligo_ai_score'].values
        preds = preds + base_preds

    # Store predictions
    test_predictions[model_name] = preds

print("\n✅ All Predictions Complete.")

Loading Models and Generating Predictions...

⚙️ Processing Baseline...


/home/michael/anaconda3/envs/tauso/lib/python3.11/site-packages/xgboost/core.py:751: UserWarning: [00:33:41] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cpu, while the input data is on: cuda:0.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


⚙️ Processing BaselineCustomId...
⚙️ Processing GA_Model...
⚙️ Processing L1_Model...
⚙️ Processing Residual...
⚙️ Processing ResidualGA...

✅ All Predictions Complete.


In [3]:
# 4. Define Evaluation Metrics
EVAL_GROUP = 'custom_id'
SELECT_GROUP = [CANONICAL_GENE, CELL_LINE]

def calculate_metrics(preds, y_true, eval_groups):
    spearmans, top_1_means, top_5_means = [], [], []
    for idxs in eval_groups:
        t_vals, p_vals = y_true[idxs], preds[idxs]
        corr, _ = spearmanr(t_vals, p_vals)
        if not np.isnan(corr): spearmans.append(corr)

        n = len(t_vals)
        k1, k5 = max(1, int(n * 0.01)), max(1, int(n * 0.05))

        if k5 > 0: top_5_means.append(np.mean(t_vals[np.argpartition(p_vals, -k5)[-k5:]]))
        if k1 > 0: top_1_means.append(np.mean(t_vals[np.argpartition(p_vals, -k1)[-k1:]]))

    return {
        'spearman': np.nanmedian(spearmans) if spearmans else 0.0,
        'top1_inhibition': np.nanmean(top_1_means) if top_1_means else 0.0,
        'top5_inhibition': np.nanmean(top_5_means) if top_5_means else 0.0
    }

# Precompute Group Indices
test_df_reset = test_df.reset_index(drop=True)
all_test_eval_idx = [group.index.values for _, group in test_df_reset.groupby(EVAL_GROUP)]
all_test_select_idx = [group.index.values for _, group in test_df_reset.groupby(SELECT_GROUP)]

# Calculate Metrics for ALL Models dynamically
all_metrics = {}
for model_name, preds in test_predictions.items():
    all_metrics[model_name] = {
        'eval_group': calculate_metrics(preds, y_test_np, all_test_eval_idx),
        'select_group': calculate_metrics(preds, y_test_np, all_test_select_idx)
    }

# 5. Print Dynamic Comparison
model_names = list(test_predictions.keys())
separator_length = 38 + (15 * len(model_names)) # Adjust table width based on model count

print("\n" + "=" * separator_length)
print("🌍 MODEL COMPARISON (TEST SET ONLY)")
print("=" * separator_length)

# Dynamic Header
header = f"{'Metric':<35} | " + " | ".join([f"{name:<12}" for name in model_names])
print(header)
print("-" * separator_length)

# Helper function to print a row dynamically for all models
def print_metric_row(label, metric_key, group_type, is_percent=False):
    print(label)
    row_str = f"{'':<35} | "
    vals = []
    for name in model_names:
        val = all_metrics[name][group_type][metric_key]
        if is_percent:
            vals.append(f"{val:<12.2f}")
        else:
            vals.append(f"{val:<12.4f}")
    print(row_str + " | ".join(vals))

# Group: custom_id
print_metric_row(f"Median Spearman ({EVAL_GROUP}):", 'spearman', 'eval_group')
print_metric_row(f"Avg Top 1% Inhib ({EVAL_GROUP}):", 'top1_inhibition', 'eval_group', is_percent=True)

print("-" * separator_length)

# Group: Canonical Gene + Cell Line
print_metric_row(f"Median Spearman (Gene/Cell):", 'spearman', 'select_group')
print_metric_row(f"Avg Top 1% Inhib (Gene/Cell):", 'top1_inhibition', 'select_group', is_percent=True)

print("=" * separator_length + "\n")

/tmp/ipykernel_194977/3874759026.py:9: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = spearmanr(t_vals, p_vals)



🌍 MODEL COMPARISON (TEST SET ONLY)
Metric                              | Baseline     | BaselineCustomId | GA_Model     | L1_Model     | Residual     | ResidualGA  
--------------------------------------------------------------------------------------------------------------------------------
Median Spearman (custom_id):
                                    | 0.4144       | 0.4245       | 0.4411       | 0.4102       | 0.4803       | 0.4891      
Avg Top 1% Inhib (custom_id):
                                    | 70.60        | 69.02        | 71.03        | 65.11        | 69.94        | 70.38       
--------------------------------------------------------------------------------------------------------------------------------
Median Spearman (Gene/Cell):
                                    | 0.3656       | 0.3810       | 0.4176       | 0.3640       | 0.4289       | 0.4159      
Avg Top 1% Inhib (Gene/Cell):
                                    | 69.22        | 70.38        | 74.81       

In [4]:
from notebooks.models.utility import load_and_validate_final_data
from tauso.data.consts import INHIBITION, CANONICAL_GENE, CELL_LINE
import pandas as pd
import numpy as np
import cupy as cp
import xgboost as xgb
from scipy.stats import spearmanr
from pathlib import Path

# 1. Load Data
final_data, all_features = load_and_validate_final_data(version='oligo', load_competition=True)
test_df = final_data[final_data['split'] == 'test'].copy()

X_test_np = test_df[all_features].values
y_test_np = test_df[INHIBITION].values

# Transfer to GPU
print("Transferring full matrices to GPU (cupy)...")
X_test_gpu = cp.array(X_test_np)
print("GPU Transfer Complete.")

seen_model_dir = Path('/home/michael/career/tauso_article/tauso_source2/notebooks/models/SeenOligoModel')

# 1. Define models to evaluate: { "Display Name": "filename.json" }
model_files = {
    'Baseline': 'ModelOligoGlobalL2.json',
    'BaselineCustomId': 'ModelOligoGlobalL2CustomId.json',
    'GA_Model': 'ModelOligoGlobalL2_GA.json',
    'L1_Model': 'ModelOligoGlobalL1.json',
    'Residual': 'GlobalResidualModel.json',
    'ResidualGA': 'GlobalResidualModelGA.json',
}

models = {}
test_predictions = {}

print("Loading Models and Generating Predictions...\n")

for model_name, filename in model_files.items():
    print(f"⚙️ Processing {model_name}...")

    # 2. Load Model
    booster = xgb.Booster()
    booster.load_model(str(seen_model_dir / filename))
    models[model_name] = booster

    # 3. Extract specific features required by this model
    feature_names = booster.feature_names
    feature_idxs = [all_features.index(f) for f in feature_names]

    # 4. Subset GPU data and predict
    X_test_gpu_subset = X_test_gpu[:, feature_idxs]
    preds = booster.inplace_predict(X_test_gpu_subset)

    if hasattr(preds, 'get'):
        preds = preds.get()

    if  'Residual' in model_name:
        base_preds = test_df['oligo_ai_score'].values
        preds = preds + base_preds

    # Store predictions
    test_predictions[model_name] = preds

# ADD OLIGO AI SCORE AS A COMPARISON BASELINE
print("⚙️ Processing OligoAI_Score (Feature Baseline)...")
test_predictions['OligoAI_Score'] = test_df['oligo_ai_score'].values

print("\n✅ All Predictions Complete.")

Transferring full matrices to GPU (cupy)...
GPU Transfer Complete.
Loading Models and Generating Predictions...

⚙️ Processing Baseline...
⚙️ Processing BaselineCustomId...
⚙️ Processing GA_Model...
⚙️ Processing L1_Model...
⚙️ Processing Residual...
⚙️ Processing ResidualGA...
⚙️ Processing OligoAI_Score (Feature Baseline)...

✅ All Predictions Complete.


In [5]:
# 4. Define Evaluation Metrics
EVAL_GROUP = 'custom_id'
SELECT_GROUP = [CANONICAL_GENE, CELL_LINE]

def calculate_metrics(preds, y_true, eval_groups):
    spearmans, top_1_meds, top_5_meds = [], [], []
    for idxs in eval_groups:
        t_vals, p_vals = y_true[idxs], preds[idxs]
        corr, _ = spearmanr(t_vals, p_vals)
        if not np.isnan(corr): spearmans.append(corr)

        n = len(t_vals)
        k1, k5 = max(1, int(n * 0.01)), max(1, int(n * 0.05))

        # Replaced mean with median
        if k5 > 0: top_5_meds.append(np.median(t_vals[np.argpartition(p_vals, -k5)[-k5:]]))
        if k1 > 0: top_1_meds.append(np.median(t_vals[np.argpartition(p_vals, -k1)[-k1:]]))

    return {
        'spearman': np.nanmedian(spearmans) if spearmans else 0.0,
        'top1_target': np.nanmedian(top_1_meds) if top_1_meds else 0.0,
        'top5_target': np.nanmedian(top_5_meds) if top_5_meds else 0.0
    }

# Precompute Group Indices
test_df_reset = test_df.reset_index(drop=True)
all_test_eval_idx = [group.index.values for _, group in test_df_reset.groupby(EVAL_GROUP)]
all_test_select_idx = [group.index.values for _, group in test_df_reset.groupby(SELECT_GROUP)]

# Calculate Metrics for ALL Models dynamically
all_metrics = {}
for model_name, preds in test_predictions.items():
    all_metrics[model_name] = {
        'eval_group': calculate_metrics(preds, y_test_np, all_test_eval_idx),
        'select_group': calculate_metrics(preds, y_test_np, all_test_select_idx)
    }

/tmp/ipykernel_194977/2666579134.py:9: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = spearmanr(t_vals, p_vals)


In [6]:
# 5. Print Dynamic Comparison
model_names = list(test_predictions.keys())
col_width = 15
separator_length = 38 + (col_width * len(model_names))

print("\n" + "=" * separator_length)
print("🌍 MODEL COMPARISON (TEST SET ONLY)")
print("=" * separator_length)

# Dynamic Header
header = f"{'Metric':<35} | " + " | ".join([f"{name:<{col_width - 3}}" for name in model_names])
print(header)
print("-" * separator_length)


def print_metric_row(label, metric_key, group_type, is_percent=False):
    print(label)
    row_str = f"{'':<35} | "
    vals = []
    for name in model_names:
        val = all_metrics[name][group_type][metric_key]
        if is_percent:
            vals.append(f"{val:<{col_width - 3}.2f}")
        else:
            vals.append(f"{val:<{col_width - 3}.4f}")
    print(row_str + " | ".join(vals))


print_metric_row(f"Median Spearman ({EVAL_GROUP}):", 'spearman', 'eval_group')
print_metric_row(f"Median Top 1% Target ({EVAL_GROUP}):", 'top1_target', 'eval_group', is_percent=True)

print("-" * separator_length)

print_metric_row(f"Median Spearman (Gene/Cell):", 'spearman', 'select_group')
print_metric_row(f"Median Top 1% Target (Gene/Cell):", 'top1_target', 'select_group', is_percent=True)

print("=" * separator_length + "\n")

# 6. Specific Evaluation Block: PSD3 Top 1% Medians by custom_id
print("\n" + "=" * separator_length)
print("🧬 PSD3 GENE EVALUATION (Median Top 1% Target by custom_id)")
print("=" * separator_length)

psd3_mask = test_df_reset[CANONICAL_GENE] == 'PSD3'
psd3_df = test_df_reset[psd3_mask].reset_index(drop=True)

if len(psd3_df) == 0:
    print("⚠️ No PSD3 data found in the test set.")
else:
    psd3_eval_idx = [group.index.values for _, group in psd3_df.groupby(EVAL_GROUP)]
    psd3_custom_ids = [name for name, _ in psd3_df.groupby(EVAL_GROUP)]

    header_psd3 = f"{'custom_id':<35} | " + " | ".join([f"{name:<{col_width - 3}}" for name in model_names])
    print(header_psd3)
    print("-" * separator_length)

    for i, idxs in enumerate(psd3_eval_idx):
        c_id = psd3_custom_ids[i]
        c_id_display = c_id[:32] + "..." if len(c_id) > 35 else c_id
        row_str = f"{c_id_display:<35} | "
        vals = []

        # Target values specifically for these PSD3 instances
        t_vals = psd3_df[INHIBITION].values[idxs]
        n = len(t_vals)
        k1 = max(1, int(n * 0.01))

        for name in model_names:
            # Filter predictions down to PSD3 rows, then subset for the custom_id
            p_vals = test_predictions[name][psd3_mask][idxs]

            if k1 > 0:
                top1_med = np.median(t_vals[np.argpartition(p_vals, -k1)[-k1:]])
                vals.append(f"{top1_med:<{col_width - 3}.2f}")
            else:
                vals.append(f"{'N/A':<{col_width - 3}}")

        print(row_str + " | ".join(vals))

print("=" * separator_length + "\n")


🌍 MODEL COMPARISON (TEST SET ONLY)
Metric                              | Baseline     | BaselineCustomId | GA_Model     | L1_Model     | Residual     | ResidualGA   | OligoAI_Score
-----------------------------------------------------------------------------------------------------------------------------------------------
Median Spearman (custom_id):
                                    | 0.4144       | 0.4245       | 0.4411       | 0.4102       | 0.4803       | 0.4891       | 0.4455      
Median Top 1% Target (custom_id):
                                    | 78.00        | 76.00        | 80.00        | 73.00        | 76.00        | 77.00        | 76.00       
-----------------------------------------------------------------------------------------------------------------------------------------------
Median Spearman (Gene/Cell):
                                    | 0.3656       | 0.3810       | 0.4176       | 0.3640       | 0.4289       | 0.4159       | 0.3689      
Median Top 1% T